# 02. QLoRA — 4비트 양자화로 메모리 절약

**QLoRA** 는 모델 가중치를 4비트로 **양자화(quantization)** 해 메모리에 올린 뒤 LoRA를 적용하는 기법입니다.
같은 GPU로 훨씬 큰 모델을 파인튜닝할 수 있어, 제한된 자원에서 LLM을 다루는 표준이 되었습니다.

1. 4비트 양자화 설정 (bitsandbytes)
2. 양자화 모델 로드 + kbit 학습 준비
3. LoRA 적용 및 학습

> 4비트 양자화(bitsandbytes)는 CUDA GPU가 필요합니다.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset

print('GPU:', torch.cuda.is_available())
model_name = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'

## 1. 4비트 양자화 설정

`BitsAndBytesConfig`로 4비트 로딩을 정의합니다. `nf4`는 정규분포 가중치에 최적화된 4비트 자료형이고,
계산은 더 높은 정밀도(bf16/fp16)로 수행합니다.

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
print('양자화 설정 완료')

## 2. 양자화 모델 로드 + kbit 학습 준비

4비트로 모델을 불러오고, `prepare_model_for_kbit_training`으로 양자화 모델을 학습 가능한 상태로 준비합니다.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(model_name, quantization_config=bnb_config)
model = prepare_model_for_kbit_training(model)
print('4비트 모델 로드 완료')

## 3. LoRA 적용 및 학습

양자화된 모델 위에 LoRA를 얹어 학습합니다 (= QLoRA).

In [ ]:
lora_config = LoraConfig(
    r=16, lora_alpha=32,
    target_modules=['q_proj', 'v_proj'],
    lora_dropout=0.05, bias='none', task_type='CAUSAL_LM',
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

dataset = load_dataset('tatsu-lab/alpaca', split='train').shuffle(seed=42).select(range(300))
def to_text(ex):
    p = f"### Instruction:\n{ex['instruction']}\n"
    if ex.get('input'): p += f"### Input:\n{ex['input']}\n"
    return {'text': p + f"### Response:\n{ex['output']}"}
dataset = dataset.map(to_text)

sft_config = SFTConfig(
    output_dir='/workspace/qlora-out',
    num_train_epochs=1, per_device_train_batch_size=2,
    gradient_accumulation_steps=4, learning_rate=2e-4,
    logging_steps=20, max_length=512, report_to='none',
)
trainer = SFTTrainer(model=model, args=sft_config, train_dataset=dataset)
trainer.train()
print('QLoRA 파인튜닝 완료')

### 정리

- 4비트 양자화로 모델 메모리를 크게 줄이고, 그 위에 LoRA를 적용해 QLoRA로 파인튜닝했습니다.
- 같은 GPU로 더 큰 모델을 다룰 수 있어, 제한된 자원에서 LLM 파인튜닝의 표준 방법입니다.
- 다음 노트북에서는 파인튜닝 대신, 외부 지식을 검색해 활용하는 **RAG**를 다룹니다.